# GitHub Repository Security Analysis

Batch-collects GitHub repositories and scans them for security vulnerabilities.

**Pipeline:**
1. Sample 100 repos per month (2017–2025) via GitHub Search API using `pushed:` date
2. Target personal (user-owned) repos with low star counts
3. Shallow-clone each repo (`--depth 1`) — `pushed:` ensures HEAD = that month's state
4. Run Semgrep for CWE/CVE detection + pip-audit/npm audit for dependency vulnerabilities
5. Save all results incrementally to CSV

**Scope:** Python, JavaScript, TypeScript, Ruby, PHP

All key parameters are adjustable in the CONFIG cell.

## 1. Setup and Configuration

In [30]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_url = "https://github.com/EthanCui2008/AAR-Research-Paper-AI-Effects.git"
    repo_dir = "/content/AAR-Research-Paper-AI-Effects"
    if not os.path.exists(repo_dir):
        os.system(f"git clone {repo_url} {repo_dir}")
    os.chdir(repo_dir)

    # Semgrep + npm (not in apt on Colab, install via pip/nvm)
    os.system(f"{sys.executable} -m pip install -q semgrep")
    os.system("npm --version >/dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs) >/dev/null 2>&1")

os.system(f"{sys.executable} -m pip install -q -r requirements.txt")
print("Environment ready.")

Environment ready.


In [ ]:
import json
import csv
import calendar
import random
import subprocess
import shutil
import time
import threading
import logging
import signal
import atexit
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
from github import Github, RateLimitExceededException
from github.GithubException import GithubException
from tqdm.auto import tqdm
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("Imports loaded.")

In [ ]:
# ── Adjustable parameters ──────────────────────────────────────────
CONFIG = {
    # Sampling
    'repos_per_month': 100,              # repos to collect per monthly interval
    'max_stars': 50,                     # upper star limit (personal/low-star repos)
    'start_year': 2017,
    'end_year': 2025,

    # Languages currently analysed (interpreted)
    'languages': ['Python', 'JavaScript', 'TypeScript', 'Ruby', 'PHP'],

    # Scanning
    'clone_timeout': 300,                # git-clone timeout (seconds)
    'semgrep_timeout': 600,              # semgrep timeout per repo (seconds)
    'max_workers': 4,                    # parallel clone+scan threads

    # Resilience
    'max_retries': 3,                    # retry attempts for transient failures
    'retry_base_delay': 15,              # base delay in seconds (exponential backoff: 15s, 30s, 60s)
    'keepalive_interval': 300,           # GitHub API ping every 5 min (seconds)
    'csv_snapshot_interval': 5,          # create CSV snapshot every N batches

    # Paths
    'repos_dir': Path('./repos'),
    'results_dir': Path('./results'),
    'log_file': Path('./results/pipeline.log'),
}

for d in [CONFIG['repos_dir'], CONFIG['results_dir']]:
    d.mkdir(exist_ok=True)

print("Config loaded.", IN_COLAB)

In [32]:
# ── Quick test run ─────────────────────────────────────────────────
# Flip TEST_MODE to True and run all cells — uses a tiny sample and
# separate output dir so it won't touch the real results.
TEST_MODE = False

if TEST_MODE:
    CONFIG['repos_per_month'] = 5
    CONFIG['results_dir'] = Path('./results_test')
    CONFIG['results_dir'].mkdir(exist_ok=True)
    print(f"TEST MODE: repos_per_month={CONFIG['repos_per_month']}, output -> {CONFIG['results_dir']}")

In [ ]:
GITHUB_TOKEN = None

if IN_COLAB:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

if not GITHUB_TOKEN:
    load_dotenv('keys.env')
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

if not GITHUB_TOKEN:
    raise ValueError(
        "GitHub token not found!\n"
        "  Local : create keys.env with GITHUB_TOKEN=ghp_...\n"
        "  Colab : add GITHUB_TOKEN to Secrets (key icon)."
    )

github_client = Github(GITHUB_TOKEN, per_page=100)

def _get_core_rate(client):
    """Extract core rate-limit object across PyGithub versions."""
    rl = client.get_rate_limit()
    # v2.x: rl.core  |  v1.x: rl.rate  |  fallback: rl itself
    for attr in ('core', 'rate'):
        obj = getattr(rl, attr, None)
        if obj is not None and hasattr(obj, 'remaining'):
            return obj
    return rl

rate = _get_core_rate(github_client)
print(f"GitHub API — {rate.remaining}/{rate.limit} requests left, resets {rate.reset}")

GitHub API — 4965/5000 requests left, resets 2026-02-27 18:04:28+00:00


## 2. Repository Collection

Repos are fetched per monthly interval across all configured languages using
the `pushed:` date qualifier (captures repos whose last activity was in that
month). Only user-owned repos with low star counts are kept. Results are saved
to `results/repositories.csv`.

In [ ]:
def _time_intervals(start_year: int, end_year: int) -> list:
    """Generate (start_date, end_date) for each month from start_year-01 to end_year-12."""
    intervals = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            s = f"{year}-{month:02d}-01"
            last_day = calendar.monthrange(year, month)[1]
            e = f"{year}-{month:02d}-{last_day:02d}"
            intervals.append((s, e))
    return intervals


class RepositoryCollector:
    """Collects GitHub repos: monthly intervals, pushed: filter, user-owned, low stars."""

    def __init__(self, client: Github, config: dict):
        self.client = client
        self.cfg = config
        self.csv_path = config['results_dir'] / 'repositories.csv'

    def _wait_for_rate_limit(self):
        rate = _get_core_rate(self.client)
        wait = (rate.reset - datetime.utcnow()).total_seconds() + 10
        if wait > 0:
            print(f"  Rate-limited -- sleeping {wait:.0f}s ...")
            time.sleep(wait)

    def _search_interval(self, lang: str, start: str, end: str) -> list:
        """Search for user-owned repos pushed in [start, end] for one language."""
        query = f"language:{lang} stars:0..{self.cfg['max_stars']} pushed:{start}..{end}"
        repos = []
        try:
            results = self.client.search_repositories(query=query, sort='stars', order='asc')
            for repo in results:
                if repo.owner.type != 'User':
                    continue
                repos.append({
                    'full_name': repo.full_name,
                    'name': repo.name,
                    'owner': repo.owner.login,
                    'owner_type': repo.owner.type,
                    'url': repo.html_url,
                    'clone_url': repo.clone_url,
                    'language': repo.language,
                    'stars': repo.stargazers_count,
                    'forks': repo.forks_count,
                    'created_at': repo.created_at.isoformat(),
                    'pushed_at': repo.pushed_at.isoformat(),
                    'description': (repo.description or '')[:200],
                    'interval_start': start,
                    'interval_end': end,
                })
                if len(repos) >= 200:
                    break
        except RateLimitExceededException:
            self._wait_for_rate_limit()
            return self._search_interval(lang, start, end)
        except GithubException as e:
            print(f"  API error ({lang} {start}): {e.status}")
        return repos

    def collect(self) -> pd.DataFrame:
        """Fetch repos per monthly interval, sample repos_per_month from each."""
        intervals = _time_intervals(self.cfg['start_year'], self.cfg['end_year'])
        languages = self.cfg['languages']
        n_per_month = self.cfg['repos_per_month']
        rng = random.Random(42)

        sampled = []
        total_queries = len(intervals) * len(languages)
        with tqdm(total=total_queries, desc="Fetching repos", unit="query") as pbar:
            for start, end in intervals:
                pool = []
                for lang in languages:
                    pbar.set_postfix_str(f"{lang} {start} | repos collected: {len(sampled)}")
                    batch = self._search_interval(lang, start, end)
                    pool.extend(batch)
                    pbar.update(1)

                # Deduplicate within this month
                seen = set()
                unique = []
                for r in pool:
                    if r['full_name'] not in seen:
                        seen.add(r['full_name'])
                        unique.append(r)

                # Sample
                if len(unique) <= n_per_month:
                    sampled.extend(unique)
                else:
                    sampled.extend(rng.sample(unique, n_per_month))

            pbar.set_postfix_str(f"done | repos collected: {len(sampled)}")

        df = pd.DataFrame(sampled).drop_duplicates(subset='full_name').reset_index(drop=True)
        df.to_csv(self.csv_path, index=False)

        counts = df.groupby('interval_start').size()
        print(f"Saved {len(df)} repos -> {self.csv_path}")
        print(f"  Months: {len(counts)},  per-month: "
              f"min={counts.min()} / median={int(counts.median())} / max={counts.max()}")
        return df

In [ ]:
collector = RepositoryCollector(github_client, CONFIG)

# On local runs, reuse existing repositories.csv if present
_repos_csv = CONFIG['results_dir'] / 'repositories.csv'
if not IN_COLAB and _repos_csv.exists() and _repos_csv.stat().st_size > 0:
    repos_df = pd.read_csv(_repos_csv)
    print(f"Local cache hit: loaded {len(repos_df)} repos from {_repos_csv}")
else:
    repos_df = collector.collect()

In [ ]:
# ── Optional: merge an extra repositories CSV ─────────────────────
# Upload (Colab) or enter a file path (local) to add repos from a
# previous run. Press Enter with no path to skip.

# If collection was skipped, bootstrap repos_df from existing CSV or empty
if 'repos_df' not in dir() or repos_df is None:
    _existing = CONFIG['results_dir'] / 'repositories.csv'
    if _existing.exists():
        repos_df = pd.read_csv(_existing)
        print(f"Loaded {len(repos_df)} repos from {_existing}")
    else:
        repos_df = pd.DataFrame()
        print("No existing repos — starting empty.")

_before = len(repos_df)

if IN_COLAB:
    from google.colab import files
    print("Upload a repositories CSV (or click cancel to skip):")
    try:
        uploaded = files.upload()
        if uploaded:
            fname = next(iter(uploaded))
            extra_df = pd.read_csv(fname)
            repos_df = pd.concat([repos_df, extra_df], ignore_index=True)
    except Exception:
        pass
else:
    _path = input("Path to extra repositories CSV (Enter to skip): ").strip()
    if _path:
        extra_df = pd.read_csv(_path)
        repos_df = pd.concat([repos_df, extra_df], ignore_index=True)

repos_df = repos_df.drop_duplicates(subset='full_name').reset_index(drop=True)
_added = len(repos_df) - _before
print(f"Added {_added} new repos  |  Total: {len(repos_df)}")

## 3. Security & Dependency Scanning

Each repo is shallow-cloned once, then scanned with:
1. **Semgrep** (`--config auto`) for CWE/CVE detection
2. **pip-audit** / **npm audit** for dependency vulnerabilities

Results are saved incrementally to `semgrep_findings.csv` and
`deprecated_dependencies.csv`. Errors go to `scan_errors.csv`.

In [ ]:
# ── Pipeline infrastructure: logging, keep-alive, shutdown, error classification ──

class PipelineLogger:
    """Dual-output logger: writes to a file (tail -f friendly) and notebook (tqdm.write)."""

    def __init__(self, log_path: Path):
        self._logger = logging.getLogger('pipeline')
        self._logger.setLevel(logging.DEBUG)
        self._logger.handlers.clear()

        # File handler — always append so the log survives restarts
        fh = logging.FileHandler(str(log_path), encoding='utf-8')
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)s | %(message)s',
                                          datefmt='%Y-%m-%d %H:%M:%S'))
        self._logger.addHandler(fh)
        self._log_path = log_path

    def _emit(self, level_fn, msg):
        level_fn(msg)
        try:
            ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            tqdm.write(f"[{ts}] {msg}")
        except Exception:
            pass

    def info(self, msg):
        self._emit(self._logger.info, msg)

    def warn(self, msg):
        self._emit(self._logger.warning, msg)

    def error(self, msg):
        self._emit(self._logger.error, msg)

    def repo_event(self, repo: str, stage: str, msg: str,
                   duration: float = None, count: int = None):
        parts = [f"[{repo}] {stage} {msg}"]
        if duration is not None:
            parts.append(f"({duration:.1f}s)")
        if count is not None:
            parts.append(f"[{count}]")
        self.info(' '.join(parts))


class KeepAlive:
    """Background daemon thread that pings the GitHub API to keep sessions warm."""

    def __init__(self, github_client, interval: int, logger: PipelineLogger):
        self._client = github_client
        self._interval = interval
        self._logger = logger
        self._stop_event = threading.Event()
        self._thread = None

    def _run(self):
        while not self._stop_event.wait(self._interval):
            try:
                rate = _get_core_rate(self._client)
                self._logger.info(f"keepalive: {rate.remaining}/{rate.limit} requests remaining")
            except Exception as e:
                self._logger.warn(f"keepalive ping failed: {e}")

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True, name='keepalive')
        self._thread.start()
        atexit.register(self.stop)
        self._logger.info(f"keepalive started (interval={self._interval}s)")

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=5)
        self._logger.info("keepalive stopped")


class GracefulShutdown:
    """Catches SIGINT/SIGTERM, sets a flag for workers, and saves pipeline state on exit."""

    def __init__(self, logger: PipelineLogger = None):
        self._requested = False
        self._logger = logger
        self._save_callback = None  # set by RepoScanner to save pipeline_state.json
        self._lock = threading.Lock()

        # Register signal handlers (may fail in non-main threads / Jupyter)
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, self._handle)
            except (OSError, ValueError):
                pass
        atexit.register(self._on_exit)

    def _handle(self, signum, frame):
        with self._lock:
            if self._requested:
                return  # already shutting down
            self._requested = True
        if self._logger:
            self._logger.warn(f"Shutdown requested (signal {signum}) — finishing current repos and flushing...")
        if self._save_callback:
            try:
                self._save_callback()
            except Exception:
                pass

    def _on_exit(self):
        if self._save_callback and self._requested:
            try:
                self._save_callback()
            except Exception:
                pass

    @property
    def shutdown_requested(self) -> bool:
        return self._requested

    def set_save_callback(self, fn):
        self._save_callback = fn


# ── Transient vs permanent error classification ────────────────────

_TRANSIENT_PATTERNS = [
    'timeout', 'timed out', 'connection reset', 'connection refused',
    'connection abort', 'broken pipe', 'network is unreachable',
    'temporary failure', 'name resolution', 'ssl', 'eof occurred',
    '500', '502', '503', '429', 'rate limit', 'server error',
]
_PERMANENT_PATTERNS = [
    '404', 'not found', '451', 'dmca', 'repository not found',
    'authentication', 'permission denied', 'access denied', '403',
    'bad credentials', 'invalid',
]

def _is_transient(error_str: str) -> bool:
    """Return True if the error looks transient and worth retrying."""
    lower = error_str.lower()
    for pat in _PERMANENT_PATTERNS:
        if pat in lower:
            return False
    for pat in _TRANSIENT_PATTERNS:
        if pat in lower:
            return True
    # Default: treat unknown errors as transient (retry is safer for a 100h run)
    return True


print("Pipeline infrastructure loaded.")

In [ ]:
class RepoScanner:
    """Clone each repo once, run Semgrep + dependency scans, save incrementally.

    Uses a cache/ folder to track per-repo progress so scanning can be
    interrupted and resumed without re-processing already-scanned repos.
    Results are flushed to CSV immediately after each repo completes.

    Resilience features:
    - Retry with exponential backoff for transient failures
    - Per-repo CSV flush with fsync (no data sits only in memory)
    - Graceful shutdown: finishes current repos, saves position, exits
    - CSV integrity verification on resume
    - Periodic CSV snapshots as safety copies
    - Dual logging to file (tail -f) and notebook (tqdm)
    """

    FINDING_FIELDS = [
        'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
        'rule_id', 'severity', 'message', 'file', 'line_start', 'line_end',
        'cwe', 'owasp', 'cve', 'confidence', 'category',
    ]
    DEP_FIELDS = [
        'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
        'package', 'version', 'vulnerability_id', 'severity',
        'description', 'ecosystem', 'fix_versions',
    ]

    BATCH_SIZE = 25  # repos per batch (used for snapshot intervals + logging)

    def __init__(self, config: dict, logger: PipelineLogger = None,
                 shutdown: GracefulShutdown = None):
        self.cfg = config
        self.findings_csv = config['results_dir'] / 'semgrep_findings.csv'
        self.deps_csv = config['results_dir'] / 'deprecated_dependencies.csv'
        self.errors_csv = config['results_dir'] / 'scan_errors.csv'
        self.state_file = config['results_dir'] / 'pipeline_state.json'
        self.cache_dir = config['results_dir'] / 'cache'
        self.snapshot_dir = config['results_dir'] / 'snapshots'
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.snapshot_dir.mkdir(parents=True, exist_ok=True)

        self._sg_lock = threading.Lock()
        self._dep_lock = threading.Lock()
        self._err_lock = threading.Lock()
        self._csv_lock = threading.Lock()  # for per-repo CSV flush
        self._env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8'}

        # Resilience config
        self._max_retries = config.get('max_retries', 3)
        self._retry_delay = config.get('retry_base_delay', 15)
        self._snapshot_interval = config.get('csv_snapshot_interval', 5)

        # Logger & shutdown
        self.log = logger or PipelineLogger(config.get('log_file', config['results_dir'] / 'pipeline.log'))
        self.shutdown = shutdown or GracefulShutdown(self.log)

        # Pipeline state for saving/resuming position
        self._state = {
            'batch_index': 0,
            'repos_scanned': 0,
            'repos_total': 0,
            'total_findings': 0,
            'total_vulns': 0,
            'cached_repos_count': 0,
        }

    # -- cache helpers ---------------------------------------------------
    def _cache_key(self, repo_name: str) -> Path:
        safe = repo_name.replace('/', '__')
        return self.cache_dir / f"{safe}.json"

    def _is_cached(self, repo_name: str) -> bool:
        return self._cache_key(repo_name).exists()

    def _write_cache(self, repo_name: str, findings: list, vulns: list, errors: list):
        payload = {'findings': findings, 'vulns': vulns, 'errors': errors}
        self._cache_key(repo_name).write_text(
            json.dumps(payload, ensure_ascii=False), encoding='utf-8')

    def _read_cache(self, repo_name: str) -> dict:
        return json.loads(self._cache_key(repo_name).read_text(encoding='utf-8'))

    def _get_cached_repos(self) -> set:
        return {p.stem.replace('__', '/') for p in self.cache_dir.glob('*.json')}

    # -- pipeline state save/load ----------------------------------------
    def _save_pipeline_state(self):
        """Write current pipeline position to disk for resume verification."""
        state = {**self._state, 'saved_at': datetime.now().isoformat()}
        self.state_file.write_text(json.dumps(state, indent=2), encoding='utf-8')

    def _load_pipeline_state(self) -> Optional[dict]:
        """Load last saved pipeline state, if it exists."""
        if self.state_file.exists():
            try:
                return json.loads(self.state_file.read_text(encoding='utf-8'))
            except Exception:
                return None
        return None

    # -- CSV integrity verification --------------------------------------
    def _verify_csv_integrity(self):
        """Check existing CSVs for corruption before rebuilding from cache."""
        csvs = [
            (self.findings_csv, self.FINDING_FIELDS),
            (self.deps_csv, self.DEP_FIELDS),
            (self.errors_csv, ['repo_name', 'stage', 'error']),
        ]
        for csv_path, expected_fields in csvs:
            if not csv_path.exists() or csv_path.stat().st_size == 0:
                continue
            try:
                df = pd.read_csv(csv_path, nrows=0)
                actual_cols = list(df.columns)
                if actual_cols != expected_fields:
                    self.log.warn(f"CSV header mismatch in {csv_path.name}: "
                                  f"expected {expected_fields}, got {actual_cols}")
                    self._backup_corrupted(csv_path)
                    continue
                # Try reading full file to detect truncation
                df_full = pd.read_csv(csv_path)
                self.log.info(f"CSV OK: {csv_path.name} ({len(df_full)} rows)")
            except Exception as e:
                self.log.warn(f"CSV corrupted: {csv_path.name} — {e}")
                self._backup_corrupted(csv_path)

    def _backup_corrupted(self, csv_path: Path):
        """Move corrupted CSV to a timestamped backup."""
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup = csv_path.with_suffix(f'.corrupted.{ts}.csv')
        shutil.move(str(csv_path), str(backup))
        self.log.warn(f"Backed up corrupted file -> {backup.name}")

    # -- CSV flush (per-repo, with fsync) --------------------------------
    def _flush_one_to_csvs(self, repo_name: str):
        """Flush a single repo's cached results to the final CSVs immediately."""
        if not self._is_cached(repo_name):
            return
        data = self._read_cache(repo_name)

        with self._csv_lock:
            if data.get('findings'):
                with open(self.findings_csv, 'a', newline='', encoding='utf-8') as f:
                    w = csv.DictWriter(f, fieldnames=self.FINDING_FIELDS)
                    for fd in data['findings']:
                        w.writerow(fd)
                    f.flush()
                    os.fsync(f.fileno())

            if data.get('vulns'):
                with open(self.deps_csv, 'a', newline='', encoding='utf-8') as f:
                    w = csv.DictWriter(f, fieldnames=self.DEP_FIELDS)
                    for v in data['vulns']:
                        w.writerow(v)
                    f.flush()
                    os.fsync(f.fileno())

            if data.get('errors'):
                with open(self.errors_csv, 'a', newline='', encoding='utf-8') as f:
                    w = csv.writer(f)
                    for e in data['errors']:
                        w.writerow(e)
                    f.flush()
                    os.fsync(f.fileno())

    def _flush_cache_to_csvs(self, repo_names: list):
        """Append cached results for given repos into the final CSVs (bulk)."""
        for name in repo_names:
            self._flush_one_to_csvs(name)

    def _init_csvs_from_cache(self, all_repo_names: list):
        """Rebuild final CSVs: write headers then flush all already-cached repos."""
        with open(self.findings_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(self.FINDING_FIELDS)
        with open(self.deps_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(self.DEP_FIELDS)
        with open(self.errors_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(['repo_name', 'stage', 'error'])
        cached = [n for n in all_repo_names if self._is_cached(n)]
        if cached:
            self._flush_cache_to_csvs(cached)
            self.log.info(f"Rebuilt CSVs from {len(cached)} cached repos")

    # -- CSV snapshots ---------------------------------------------------
    def _create_csv_snapshot(self):
        """Copy current CSVs to a timestamped snapshot directory."""
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        snap_dir = self.snapshot_dir / ts
        snap_dir.mkdir(parents=True, exist_ok=True)
        for csv_path in [self.findings_csv, self.deps_csv, self.errors_csv]:
            if csv_path.exists():
                shutil.copy2(str(csv_path), str(snap_dir / csv_path.name))
        # Also snapshot the state file
        if self.state_file.exists():
            shutil.copy2(str(self.state_file), str(snap_dir / self.state_file.name))
        self.log.info(f"CSV snapshot saved -> snapshots/{ts}/")
        self._cleanup_old_snapshots()

    def _cleanup_old_snapshots(self, keep: int = 3):
        """Keep only the N most recent snapshots."""
        snaps = sorted(self.snapshot_dir.iterdir(), reverse=True)
        for old in snaps[keep:]:
            if old.is_dir():
                shutil.rmtree(str(old), ignore_errors=True)

    def _log_error(self, repo_name: str, stage: str, error: str):
        with self._err_lock:
            with open(self.errors_csv, 'a', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow([repo_name, stage, error.replace('
', ' ').replace('', '')[:300]])

    def _warm_rules_cache(self):
        """Download semgrep rules once before parallel workers start."""
        self.log.info("Warming semgrep rules cache ...")
        tmp = self.cfg['repos_dir'] / '_semgrep_warmup'
        tmp.mkdir(parents=True, exist_ok=True)
        dummy = tmp / 'test.py'
        dummy.write_text('x = 1\n', encoding='utf-8')
        try:
            r = subprocess.run(
                ['semgrep', '--config', 'auto', '--json',
                 tmp.resolve().as_posix()],
                capture_output=True, timeout=120, env=self._env,
            )
            if r.returncode == 0:
                self.log.info("  Rules cached.")
            else:
                stderr = r.stderr.decode('utf-8', errors='replace')[:500]
                self.log.warn(f"  Cache warning (rc={r.returncode}): {stderr}")
        except Exception as e:
            self.log.warn(f"  Cache warm-up failed: {e}")
        finally:
            shutil.rmtree(tmp, ignore_errors=True)

    # -- clone (with retry) ----------------------------------------------
    def _clone(self, repo: dict) -> Optional[Path]:
        dest = self.cfg['repos_dir'] / repo['full_name'].replace('/', '_')
        if dest.exists():
            return dest
        dest.parent.mkdir(parents=True, exist_ok=True)

        for attempt in range(1, self._max_retries + 1):
            t0 = time.time()
            try:
                r = subprocess.run(
                    ['git', 'clone', '--depth', '1', repo['clone_url'], str(dest)],
                    capture_output=True, timeout=self.cfg['clone_timeout'],
                )
                elapsed = time.time() - t0

                if r.returncode == 0:
                    self.log.repo_event(repo['full_name'], 'clone', 'OK', duration=elapsed)
                    return dest

                stderr = r.stderr.decode('utf-8', errors='replace')[:200]
                error_msg = f"rc={r.returncode}: {stderr}"
                if dest.exists():
                    shutil.rmtree(dest, ignore_errors=True)

                if not _is_transient(error_msg) or attempt == self._max_retries:
                    self.log.repo_event(repo['full_name'], 'clone',
                                        f"FAIL: {error_msg}", duration=elapsed)
                    self._log_error(repo['full_name'], 'clone', error_msg)
                    return None

                delay = self._retry_delay * (2 ** (attempt - 1))
                self.log.repo_event(repo['full_name'], 'clone',
                                    f"RETRY {attempt}/{self._max_retries}: {error_msg} "
                                    f"(waiting {delay}s)", duration=elapsed)
                time.sleep(delay)

            except subprocess.TimeoutExpired:
                elapsed = time.time() - t0
                if dest.exists():
                    shutil.rmtree(dest, ignore_errors=True)
                if attempt == self._max_retries:
                    self.log.repo_event(repo['full_name'], 'clone',
                                        'FAIL: timed out', duration=elapsed)
                    self._log_error(repo['full_name'], 'clone', 'timed out')
                    return None
                delay = self._retry_delay * (2 ** (attempt - 1))
                self.log.repo_event(repo['full_name'], 'clone',
                                    f"RETRY {attempt}/{self._max_retries}: timed out "
                                    f"(waiting {delay}s)", duration=elapsed)
                time.sleep(delay)

            except Exception as e:
                elapsed = time.time() - t0
                if dest.exists():
                    shutil.rmtree(dest, ignore_errors=True)
                self.log.repo_event(repo['full_name'], 'clone',
                                    f"FAIL: {e}", duration=elapsed)
                self._log_error(repo['full_name'], 'clone', str(e))
                return None

        return None

    # -- semgrep scan (with retry) ---------------------------------------
    def _scan_semgrep(self, repo_path: Path, repo_name: str) -> list[dict]:
        posix_path = repo_path.resolve().as_posix()

        for attempt in range(1, self._max_retries + 1):
            t0 = time.time()
            try:
                r = subprocess.run(
                    ['semgrep', '--config', 'auto', '--json',
                     '--timeout', '60', '--max-target-bytes', '1000000',
                     posix_path],
                    capture_output=True,
                    timeout=self.cfg['semgrep_timeout'],
                    cwd=posix_path,
                    env=self._env,
                )
                elapsed = time.time() - t0
                stdout = r.stdout.decode('utf-8', errors='replace')
                stderr = r.stderr.decode('utf-8', errors='replace')
            except subprocess.TimeoutExpired:
                elapsed = time.time() - t0
                if attempt == self._max_retries:
                    self.log.repo_event(repo_name, 'semgrep', 'FAIL: timed out', duration=elapsed)
                    self._log_error(repo_name, 'semgrep', 'timed out')
                    return []
                delay = self._retry_delay * (2 ** (attempt - 1))
                self.log.repo_event(repo_name, 'semgrep',
                                    f"RETRY {attempt}/{self._max_retries}: timed out "
                                    f"(waiting {delay}s)", duration=elapsed)
                time.sleep(delay)
                continue
            except Exception as e:
                elapsed = time.time() - t0
                error_msg = str(e)
                if _is_transient(error_msg) and attempt < self._max_retries:
                    delay = self._retry_delay * (2 ** (attempt - 1))
                    self.log.repo_event(repo_name, 'semgrep',
                                        f"RETRY {attempt}/{self._max_retries}: {error_msg} "
                                        f"(waiting {delay}s)", duration=elapsed)
                    time.sleep(delay)
                    continue
                self.log.repo_event(repo_name, 'semgrep', f"FAIL: {e}", duration=elapsed)
                self._log_error(repo_name, 'semgrep', error_msg)
                return []

            if not stdout:
                error_msg = f"empty stdout, rc={r.returncode}, stderr={stderr[:300]}"
                # Don't retry empty stdout — likely a semgrep issue with the repo
                self.log.repo_event(repo_name, 'semgrep', f"FAIL: {error_msg}", duration=elapsed)
                self._log_error(repo_name, 'semgrep', error_msg)
                return []

            try:
                data = json.loads(stdout)
            except json.JSONDecodeError as e:
                # Don't retry JSON parse errors — output won't change
                error_msg = f"bad JSON: {e}, stdout[:200]={stdout[:200]}"
                self.log.repo_event(repo_name, 'semgrep', f"FAIL: {error_msg}", duration=elapsed)
                self._log_error(repo_name, 'semgrep', error_msg)
                return []

            for err in data.get('errors', []):
                msg = err.get('message', '') or err.get('long_msg', '') or str(err)
                self._log_error(repo_name, 'semgrep-error', msg[:300])

            findings = []
            for hit in data.get('results', []):
                meta = hit.get('extra', {}).get('metadata', {})
                cwe_raw = meta.get('cwe', [])
                cve_raw = meta.get('cve', '') or meta.get('references', '')
                findings.append({
                    'rule_id': hit.get('check_id', ''),
                    'severity': hit.get('extra', {}).get('severity', ''),
                    'message': (hit.get('extra', {}).get('message', '') or '').replace('
', ' ').replace('', '')[:300],
                    'file': hit.get('path', ''),
                    'line_start': hit.get('start', {}).get('line', 0),
                    'line_end': hit.get('end', {}).get('line', 0),
                    'cwe': ','.join(cwe_raw) if isinstance(cwe_raw, list) else str(cwe_raw),
                    'owasp': ','.join(meta.get('owasp', [])) if isinstance(meta.get('owasp'), list) else str(meta.get('owasp', '')),
                    'cve': cve_raw if isinstance(cve_raw, str) else ','.join(cve_raw),
                    'confidence': meta.get('confidence', ''),
                    'category': meta.get('category', ''),
                })

            self.log.repo_event(repo_name, 'semgrep', f'{len(findings)} findings',
                                duration=elapsed, count=len(findings))
            return findings

        return []

    # -- dependency scan (unchanged) -------------------------------------
    def _scan_python(self, req_file: Path, repo_name: str) -> list[dict]:
        vulns = []
        try:
            r = subprocess.run(
                ['pip-audit', '-r', str(req_file), '--format', 'json'],
                capture_output=True, timeout=120, env=self._env,
            )
            stdout = r.stdout.decode('utf-8', errors='replace')
            if not stdout:
                return []
            for dep in json.loads(stdout).get('dependencies', []):
                for v in dep.get('vulns', []):
                    vulns.append({
                        'package': dep.get('name', ''),
                        'version': dep.get('version', ''),
                        'vulnerability_id': v.get('id', ''),
                        'severity': v.get('fix_versions', [''])[0] if v.get('fix_versions') else '',
                        'description': (v.get('description', '') or '').replace('
', ' ').replace('', '')[:300],
                        'ecosystem': 'python',
                        'fix_versions': ','.join(v.get('fix_versions', [])),
                    })
        except json.JSONDecodeError as e:
            self._log_error(repo_name, 'pip-audit', f"bad JSON: {e}")
        except Exception as e:
            self._log_error(repo_name, 'pip-audit', str(e))
        return vulns

    def _scan_npm(self, pkg_dir: Path, repo_name: str) -> list[dict]:
        vulns = []
        try:
            r = subprocess.run(
                ['npm', 'audit', '--json'],
                capture_output=True, timeout=120,
                cwd=str(pkg_dir), env=self._env,
            )
            stdout = r.stdout.decode('utf-8', errors='replace')
            if not stdout:
                return []
            for vid, v in json.loads(stdout).get('vulnerabilities', {}).items():
                vulns.append({
                    'package': v.get('name', vid),
                    'version': v.get('range', ''),
                    'vulnerability_id': vid,
                    'severity': v.get('severity', ''),
                    'description': (v.get('title', '') or '').replace('
', ' ').replace('', '')[:300],
                    'ecosystem': 'npm',
                    'fix_versions': '',
                })
        except json.JSONDecodeError as e:
            self._log_error(repo_name, 'npm-audit', f"bad JSON: {e}")
        except Exception as e:
            self._log_error(repo_name, 'npm-audit', str(e))
        return vulns

    def _scan_dependencies(self, repo_path: Path, repo_name: str) -> list[dict]:
        vulns = []
        for p in repo_path.rglob('requirements.txt'):
            vulns.extend(self._scan_python(p, repo_name))
        for p in repo_path.rglob('package.json'):
            if 'node_modules' not in str(p):
                vulns.extend(self._scan_npm(p.parent, repo_name))
        return vulns

    # -- single-repo worker (shutdown-aware, timed, logged) --------------
    def _process_one(self, repo: dict) -> dict:
        """Clone, scan semgrep + deps, return structured result for caching."""
        result = {'findings': [], 'vulns': [], 'errors': []}
        t0 = time.time()

        try:
            # Check shutdown before clone
            if self.shutdown.shutdown_requested:
                self.log.info(f"[{repo['full_name']}] skipped (shutdown requested)")
                return result

            path = self._clone(repo)
            if path is None:
                return result

            # Check shutdown before semgrep
            if self.shutdown.shutdown_requested:
                self.log.info(f"[{repo['full_name']}] skipped semgrep (shutdown requested)")
                return result

            # Semgrep findings (enrich with repo metadata)
            raw_findings = self._scan_semgrep(path, repo['full_name'])
            for fd in raw_findings:
                fd.update({
                    'repo_name': repo['full_name'],
                    'language': repo.get('language', ''),
                    'stars': repo.get('stars', ''),
                    'interval_start': repo.get('interval_start', ''),
                    'interval_end': repo.get('interval_end', ''),
                })
            result['findings'] = raw_findings

            # Check shutdown before deps
            if self.shutdown.shutdown_requested:
                self.log.info(f"[{repo['full_name']}] skipped deps (shutdown requested)")
                return result

            # Dependency vulns (enrich with repo metadata)
            raw_vulns = self._scan_dependencies(path, repo['full_name'])
            for v in raw_vulns:
                v.update({
                    'repo_name': repo['full_name'],
                    'language': repo.get('language', ''),
                    'stars': repo.get('stars', ''),
                    'interval_start': repo.get('interval_start', ''),
                    'interval_end': repo.get('interval_end', ''),
                })
            result['vulns'] = raw_vulns

        except Exception as exc:
            result['errors'].append([repo['full_name'], 'scan', str(exc)[:300]])
            self.log.error(f"[{repo['full_name']}] scan exception: {exc}")
        finally:
            dest = self.cfg['repos_dir'] / repo['full_name'].replace('/', '_')
            shutil.rmtree(dest, ignore_errors=True)

        elapsed = time.time() - t0
        self.log.repo_event(repo['full_name'], 'done',
                            f"{len(result['findings'])} findings, "
                            f"{len(result['vulns'])} vulns",
                            duration=elapsed)
        return result

    # -- public API ------------------------------------------------------
    def scan_all(self, repos_df: pd.DataFrame):
        all_names = repos_df['full_name'].tolist()

        # Verify CSV integrity before rebuilding
        self.log.info("Verifying CSV integrity ...")
        self._verify_csv_integrity()

        # Rebuild final CSVs from any cached results
        self._init_csvs_from_cache(all_names)

        # Load previous state if resuming
        prev_state = self._load_pipeline_state()
        if prev_state:
            self.log.info(f"Previous run state: {prev_state.get('repos_scanned', '?')}/"
                          f"{prev_state.get('repos_total', '?')} repos, "
                          f"saved at {prev_state.get('saved_at', '?')}")

        # Determine which repos still need scanning
        cached = self._get_cached_repos()
        pending_df = repos_df[~repos_df['full_name'].isin(cached)].reset_index(drop=True)
        n_cached = len(repos_df) - len(pending_df)
        if n_cached:
            self.log.info(f"Resuming: {n_cached} repos already cached, {len(pending_df)} remaining")

        if pending_df.empty:
            self.log.info("All repos already scanned (cache is complete).")
            return

        self._warm_rules_cache()

        total_repos = len(repos_df)
        total_pending = len(pending_df)
        workers = self.cfg.get('max_workers', 4)
        batch_size = self.BATCH_SIZE
        total_findings = total_vulns = scanned = 0

        # Count cached stats for accurate totals
        for name in cached:
            if self._is_cached(name):
                data = self._read_cache(name)
                total_findings += len(data.get('findings', []))
                total_vulns += len(data.get('vulns', []))
        scanned = n_cached

        # Update state
        self._state.update({
            'repos_total': total_repos,
            'repos_scanned': scanned,
            'total_findings': total_findings,
            'total_vulns': total_vulns,
            'cached_repos_count': n_cached,
        })

        # Register state-save callback for graceful shutdown
        self.shutdown.set_save_callback(self._save_pipeline_state)

        # Process in batches
        n_batches = (total_pending + batch_size - 1) // batch_size
        self.log.info(f"Scanning {total_pending} repos in {n_batches} batches of ~{batch_size} "
                      f"({workers} workers)")
        pipeline_start = time.time()
        repo_times = []  # for ETA calculation

        with tqdm(total=total_repos, initial=n_cached,
                  desc=f"Scanning ({workers} workers)", unit="repo") as pbar:
            for batch_idx in range(n_batches):
                # Check shutdown before starting batch
                if self.shutdown.shutdown_requested:
                    self.log.warn(f"Shutdown: stopping before batch {batch_idx + 1}/{n_batches}")
                    break

                batch_start = time.time()
                start = batch_idx * batch_size
                end = min(start + batch_size, total_pending)
                batch_df = pending_df.iloc[start:end]
                batch_repos_done = 0

                with ThreadPoolExecutor(max_workers=workers) as pool:
                    futures = {}
                    for _, row in batch_df.iterrows():
                        f = pool.submit(self._process_one, row.to_dict())
                        futures[f] = row.to_dict()

                    for f in as_completed(futures):
                        repo = futures[f]
                        name = repo['full_name']
                        try:
                            result = f.result()
                            # Write per-repo cache file
                            self._write_cache(name, result['findings'],
                                              result['vulns'], result['errors'])
                            # Immediately flush this repo's results to CSVs
                            self._flush_one_to_csvs(name)
                            total_findings += len(result['findings'])
                            total_vulns += len(result['vulns'])
                        except Exception as exc:
                            self._write_cache(name, [], [],
                                              [[name, 'future', str(exc)[:300]]])
                            self._flush_one_to_csvs(name)
                            self.log.error(f"[{name}] future exception: {exc}")

                        scanned += 1
                        batch_repos_done += 1
                        pbar.set_postfix(
                            scanned=f"{scanned}/{total_repos}",
                            remaining=total_repos - scanned,
                            findings=total_findings,
                            vulns=total_vulns,
                        )
                        pbar.update(1)

                # Batch summary with ETA
                batch_elapsed = time.time() - batch_start
                repo_times.append(batch_elapsed / max(batch_repos_done, 1))
                avg_time = sum(repo_times[-20:]) / len(repo_times[-20:])  # rolling avg
                remaining_repos = total_repos - scanned
                eta_hours = (remaining_repos * avg_time) / 3600

                self.log.info(
                    f"BATCH {batch_idx + 1}/{n_batches} complete | "
                    f"{scanned}/{total_repos} repos | "
                    f"{total_findings} findings | {total_vulns} vulns | "
                    f"ETA ~{eta_hours:.1f}h"
                )

                # Update and save pipeline state
                self._state.update({
                    'batch_index': batch_idx + 1,
                    'repos_scanned': scanned,
                    'total_findings': total_findings,
                    'total_vulns': total_vulns,
                    'cached_repos_count': scanned,
                })
                self._save_pipeline_state()

                # Periodic CSV snapshot
                if (batch_idx + 1) % self._snapshot_interval == 0:
                    self._create_csv_snapshot()

        # Final state save
        self._save_pipeline_state()

        # Final summary
        total_elapsed = time.time() - pipeline_start
        try:
            errs_df = pd.read_csv(self.errors_csv)
            n_errors = len(errs_df)
        except Exception:
            n_errors = 0

        self.log.info(
            f"Done: {total_findings} findings, {total_vulns} dep vulns "
            f"across {scanned} repos in {total_elapsed / 3600:.1f}h"
        )
        self.log.info(f"  Errors: {n_errors}  (see {self.errors_csv})")
        self.log.info(f"  {self.findings_csv}")
        self.log.info(f"  {self.deps_csv}")

        if self.shutdown.shutdown_requested:
            self.log.warn("Pipeline stopped early due to shutdown request. "
                          "Re-run to resume from where it left off.")

In [ ]:
# Set up pipeline infrastructure
pipeline_log = PipelineLogger(CONFIG['log_file'])
shutdown_handler = GracefulShutdown(pipeline_log)
keepalive = KeepAlive(github_client, CONFIG.get('keepalive_interval', 300), pipeline_log)

# Start background keep-alive
keepalive.start()

# Run scanner
scanner = RepoScanner(CONFIG, logger=pipeline_log, shutdown=shutdown_handler)
scanner.scan_all(repos_df)

# Cleanup
keepalive.stop()
pipeline_log.info("Pipeline finished.")

## 4. Output Summary

In [ ]:
print("Output CSVs:")
for p in sorted(CONFIG['results_dir'].glob('*.csv')):
    size = p.stat().st_size
    print(f"  {p.name:40s}  {size:>10,} bytes")